# Mystery & Sci-Fi — One-Click Colab Run

This is the straight-through runner for the first six-scene test. It does the setup, syncs the latest production code, checks your Colab Secret, prepares the episode, writes narration and timed captions, and creates the Google Flow manifest in one cell.

**Use:** Open in Google Colab and run the single code cell once. No separate setup cell, pasted patch, or Prepare button.

- Uses your existing Google Drive workspace and resumes files from the previous attempt.
- Uses the six-scene premise already selected.
- Keeps Gemini image API generation **off**; this does not spend separate image-generation API credits.
- Saves a persistent log in the project folder, including the last output if a run fails.
- Google Flow video generation is still a manual step after preparation; this notebook will not spend Flow credits automatically.


In [ ]:
import os, sys, json, time, shutil, subprocess, traceback
from pathlib import Path
from google.colab import drive, userdata
from IPython.display import display, Markdown

# Fixed for the first six-scene test; edit these values only when making a different episode.
TOPIC = "An abandoned coastal radio station receives a warning about an event that has not happened yet."
SCENE_COUNT = 6
VOICE = "en-US-ChristopherNeural"
WHISPER_MODEL = "base.en"
WHISPER_DEVICE = "cpu"
PROJECT_ID = "20261120_123020_mystery_scifi_An_abandoned_coastal_radio_station_recei"
REPO_URL = "https://github.com/topboyasian-stack/gta6-autoclipper.git"
ROOT = Path("/content/gta6-autoclipper")

try:
    if not Path("/content/drive/MyDrive").is_dir():
        print("Connecting Google Drive…", flush=True)
        drive.mount("/content/drive")

    WORKSPACE = Path("/content/drive/MyDrive/GTA6_AutoClipper")
    PROJECT_DIR = WORKSPACE / "projects" / PROJECT_ID
    OUTPUT_DIR = PROJECT_DIR / "mystery_scifi"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    LOG_PATH = OUTPUT_DIR / "mystery_run.log"

    def note(message):
        stamp = time.strftime("%Y-%m-%d %H:%M:%S")
        line = f"[{stamp}] {message}"
        print(line, flush=True)
        with LOG_PATH.open("a", encoding="utf-8") as log:
            log.write(line + "\n")
            log.flush()

    with LOG_PATH.open("a", encoding="utf-8") as log:
        log.write("\n\n===== One-click six-scene run: " + time.strftime("%Y-%m-%d %H:%M:%S") + " =====\n")

    note("Persistent log: " + str(LOG_PATH))
    note("Project output: " + str(OUTPUT_DIR))

    try:
        api_key = userdata.get("GEMINI_API_KEY")
    except Exception:
        api_key = None
    if not api_key or len(str(api_key).strip()) < 10:
        raise RuntimeError(
            "Colab Secret GEMINI_API_KEY is missing or inaccessible. Add the Google AI Studio API key "
            "in Colab Secrets and enable notebook access, then rerun this same single cell."
        )
    api_key = str(api_key).strip()

    def run_logged(label, command, env=None):
        note("START: " + label)
        with LOG_PATH.open("a", encoding="utf-8") as log:
            log.write("COMMAND: " + " ".join(str(part) for part in command) + "\n")
            log.flush()
            process = subprocess.Popen(
                [str(part) for part in command],
                cwd=str(ROOT) if ROOT.is_dir() else None,
                env=env,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                bufsize=1,
                errors="replace",
            )
            log.write("PROCESS_ID: " + str(process.pid) + "\n")
            log.flush()
            print("Process started: " + str(process.pid), flush=True)
            for line in process.stdout:
                log.write(line)
                log.flush()
                print(line, end="", flush=True)
            return_code = process.wait()
            log.write("EXIT_CODE: " + str(return_code) + "\n")
            log.flush()
        if return_code != 0:
            raise RuntimeError(f"{label} failed with exit code {return_code}. See {LOG_PATH}")
        note("DONE: " + label)

    if (ROOT / ".git").is_dir():
        run_logged("Sync latest GitHub production code", ["git", "-C", str(ROOT), "fetch", "--depth=1", "origin", "main"])
        run_logged("Select the latest tested main commit", ["git", "-C", str(ROOT), "reset", "--hard", "FETCH_HEAD"])
    else:
        if ROOT.exists():
            shutil.rmtree(ROOT)
        run_logged("Download production code", ["git", "clone", "--depth", "1", REPO_URL, str(ROOT)])

    # Reuse a valid six-scene plan. Back up mismatched plans instead of silently using the wrong count.
    plan_path = OUTPUT_DIR / "episode.json"
    if plan_path.is_file():
        try:
            previous = json.loads(plan_path.read_text(encoding="utf-8"))
            previous_count = len(previous.get("scenes", []))
        except Exception:
            previous_count = -1
        if previous_count != SCENE_COUNT:
            backup = OUTPUT_DIR / ("episode.backup.before_six_scene_" + time.strftime("%Y%m%d_%H%M%S") + ".json")
            shutil.copy2(plan_path, backup)
            plan_path.unlink()
            note(f"Previous plan had {previous_count} scenes or was unreadable; backed it up to {backup}.")
        else:
            note("Found a saved six-scene plan; the CLI can resume from it without replacing it.")

    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    (PROJECT_DIR / "project_state.json").write_text(
        json.dumps({
            "project_name": PROJECT_ID,
            "mode": "Mystery & Sci-Fi Automation",
            "story_title": TOPIC[:160],
            "mystery_output": str(OUTPUT_DIR),
            "updated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            "setup": "one-click-colab",
        }, indent=2),
        encoding="utf-8",
    )

    if not shutil.which("ffmpeg") or not shutil.which("ffprobe"):
        run_logged(
            "Install FFmpeg",
            ["bash", "-lc", "apt-get update -qq && apt-get install -y -qq ffmpeg fonts-dejavu-core"],
        )
    else:
        note("FFmpeg and ffprobe are available.")

    note("Install/check production dependencies; image-generation API stays disabled.")
    run_logged("Install Python dependencies", [
        sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check",
        "google-genai>=2.12.1,<3",
        "pydantic>=2.7,<3",
        "python-dotenv>=1.0,<2",
        "edge-tts>=7.0,<9",
        "faster-whisper>=1.0,<2",
        "av==17.1.0",
        "Pillow>=10,<13",
    ])

    env = os.environ.copy()
    env.update({
        "PYTHONUNBUFFERED": "1",
        "GEMINI_API_KEY": api_key,
        "ALLOW_PAID_IMAGE_API": "false",
        "MYSTERY_WHISPER_DEVICE": WHISPER_DEVICE,
        "MYSTERY_WHISPER_MODEL": WHISPER_MODEL,
        "MYSTERY_NARRATOR_VOICE": VOICE,
        "MYSTERY_MUSIC_TRACK": str(OUTPUT_DIR / "no_music_selected.mp3"),
    })
    command = [
        sys.executable, str(ROOT / "main.py"),
        "--stage", "prepare",
        "--topic", TOPIC,
        "--scene-count", str(SCENE_COUNT),
        "--output", str(OUTPUT_DIR),
        "--voice", VOICE,
        "--whisper-model", WHISPER_MODEL,
        "--whisper-device", WHISPER_DEVICE,
        "--max-scenes-per-flow-run", "8",
    ]
    # Deliberately omit --allow-paid-images.
    run_logged("Prepare six-scene episode, narration, captions, and Flow manifest", command, env=env)

    required_files = [
        OUTPUT_DIR / "episode.json",
        OUTPUT_DIR / "episode_brief.md",
        OUTPUT_DIR / "assets_generation_plan.md",
        OUTPUT_DIR / "audio" / "narration_manifest.json",
        OUTPUT_DIR / "captions" / "subtitles.srt",
        OUTPUT_DIR / "flow_upload_manifest.json",
        OUTPUT_DIR / "flow_upload_manifest.txt",
        OUTPUT_DIR / "flow_master_prompt.md",
    ]
    missing = [str(path) for path in required_files if not path.is_file()]
    if missing:
        raise RuntimeError("The process exited successfully but required output files are missing: " + ", ".join(missing))

    episode = json.loads((OUTPUT_DIR / "episode.json").read_text(encoding="utf-8"))
    if len(episode.get("scenes", [])) != SCENE_COUNT:
        raise RuntimeError(f"Expected {SCENE_COUNT} scenes, found {len(episode.get('scenes', []))}.")

    note("SUCCESS: six-scene episode plan, narration, timed captions, and Flow manifest verified.")
    display(Markdown(
        "## Six-scene preparation complete\n\n"
        f"**Episode:** {episode.get('title', 'Untitled')}\n\n"
        f"**Scenes:** {len(episode['scenes'])}\n\n"
        f"**Project folder:** {OUTPUT_DIR}\n\n"
        f"**Flow manifest:** {OUTPUT_DIR / 'flow_upload_manifest.txt'}\n\n"
        f"**Master prompt:** {OUTPUT_DIR / 'flow_master_prompt.md'}\n\n"
        f"**Narration:** {OUTPUT_DIR / 'audio'}\n\n"
        f"**Captions:** {OUTPUT_DIR / 'captions' / 'subtitles.srt'}\n\n"
        f"**Persistent run log:** {LOG_PATH}\n\n"
        "Image API calls were disabled. Video creation in Google Flow remains a manual stage; the manifest "
        "lists the six clips in order."
    ))

except Exception as exc:
    error = f"FAILED: {type(exc).__name__}: {exc}"
    print("\n" + error, flush=True)
    try:
        with LOG_PATH.open("a", encoding="utf-8") as log:
            log.write("\n" + error + "\n")
            log.write(traceback.format_exc() + "\n")
            log.flush()
        tail = LOG_PATH.read_text(encoding="utf-8", errors="replace").splitlines()[-35:]
        print("\nLast log lines:", flush=True)
        print("\n".join(tail), flush=True)
        print("\nFull persistent log: " + str(LOG_PATH), flush=True)
    except Exception:
        print(traceback.format_exc(), flush=True)
